In [2]:
import sys
import os

# Projenin ana klasörünü Python arama yoluna ekliyoruz
sys.path.append(os.path.abspath(".."))

# Artık modülleri sorunsuz import edebilirsin
from src.data_loader import DataLoader
from src.features import FeatureEngineer
from src.preprocessing import Preprocessor

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier

# 1. Proje ana dizinini ekle (ModuleNotFoundError Çözümü)
sys.path.append(os.path.abspath(".."))

from src.data_loader import DataLoader
from src.features import FeatureEngineer
from src.preprocessing import Preprocessor

# 2. Verileri Yükle ve Ön İşlemlerden Geçir
loader = DataLoader(raw_data_dir="../data/raw", processed_data_dir="../data/processed")
train_df, test_df = loader.load_raw_data()

fe = FeatureEngineer()
train_fe = fe.create_features(train_df)
test_fe = fe.create_features(test_df)

prep = Preprocessor()
train_proc = prep.fit_transform(train_fe)
test_proc = prep.transform(test_fe)

# 3. Modeli En İyi Hiperparametrelerle Eğit (Gradient Boosting Tuned)
X_train = train_proc.drop(columns=['Survived'])
y_train = train_proc['Survived']
X_test = test_proc.copy()

best_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
best_model.fit(X_train, y_train)

# 4. Test Seti Tahminlerini Al
test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)[:, 1]

# 5. Tahminleri Sonuç Tablosunda Birleştir
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Name': test_df['Name'],
    'Pclass': test_df['Pclass'],
    'Sex': test_df['Sex'],
    'Survived_Prediction': test_preds,
    'Survival_Probability (%)': (test_probs * 100).round(2)
})

# 6. ÇIKTI 1: İlk 10 Test Yolcusunun Tahminleri
print("=" * 70)
print("📊 TEST VERİSETİ TAHMİN ÖRNEKLERİ (İLK 10 YOLCU)")
print("=" * 70)
display(submission_df.head(10))

# 7. ÇIKTI 2: Genel Tahmin İstatistiği
print("\n" + "=" * 70)
print("📈 TEST VERİSETİ TAHMİN DAĞILIMI")
print("=" * 70)
summary = submission_df['Survived_Prediction'].value_counts(normalize=True) * 100
print(f"Ölen Tahmini (0)    : %{summary[0]:.2f} ({sum(test_preds == 0)} kişi)")
print(f"Kurtulan Tahmini (1): %{summary[1]:.2f} ({sum(test_preds == 1)} kişi)")

# 8. ÇIKTI 3: Tahmin Dağılım Grafiği
plt.figure(figsize=(6, 4))
sns.countplot(data=submission_df, x='Survived_Prediction', palette='Set2')
plt.title('Test Seti Hayatta Kalma Tahmini Dağılımı')
plt.xticks([0, 1], ['Öldü (0)', 'Kurtuldu (1)'])
plt.ylabel('Yolcu Sayısı')
plt.show()